# Download chips from Google Earth Engine (Landsat 8) for India DHS 2015

First run will ask you to authenticate Google Earth Engine (GEE).

In [ ]:
!pip -q install earthengine-api geemap tqdm

import os, math, time, requests
import pandas as pd
from tqdm import tqdm
import geemap
import ee

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive"

In [ ]:
CLUSTERS_CSV = os.path.join(DRIVE_ROOT, "IN2015_clusters_nightlights.csv") #Cluster-level labels
OUT_DIR = os.path.join(DRIVE_ROOT, "india_2015_images", "gee_chips_2015_random20k")
META_OUT = os.path.join(OUT_DIR, "download_status.csv")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np

def make_sample_points_from_clusters(
    clusters: pd.DataFrame,
    n_per_cluster: int = 25,
    box_km: int = 5,
    seed: int = 0
):
    rng = np.random.default_rng(seed)
    rows = []

    for row in clusters.itertuples(index=False):
        cid = int(row.cluster_id)
        lat = float(row.cluster_lat)
        lon = float(row.cluster_lon)

        d = box_km / 111.0

        # Generate random sample points around the cluster center
        lats = lat + rng.uniform(-d, d, size=n_per_cluster)
        lons = lon + rng.uniform(-d, d, size=n_per_cluster) / max(
            0.1, np.cos(np.deg2rad(lat))
        )

        for i, (la, lo) in enumerate(zip(lats, lons)):
            rows.append((cid, i, la, lo))

    return pd.DataFrame(
        rows,
        columns=["cluster_id", "sample_id", "lat", "lon"]
    )

clusters = pd.read_csv(CLUSTERS_CSV)
locs = make_sample_points_from_clusters(
    clusters,
    n_per_cluster=25,
    box_km=5,
    seed=42
)

locs.head()

### Authenticate Earth Engine


In [ ]:
ee.Authenticate()

# Initialize with your project id
ee.Initialize(project="project-5ac8fb15-99fe-49bf-9b9")

## Landsat 8 (2015) composite and chip download

In [ ]:
def mask_l8_sr(img):
    qa = img.select("QA_PIXEL")

    # QA_PIXEL bit flags: cloud, shadow, snow, cirrus
    cloud  = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    snow   = qa.bitwiseAnd(1 << 5).neq(0)
    cirrus = qa.bitwiseAnd(1 << 7).neq(0)

    mask = cloud.Or(shadow).Or(snow).Or(cirrus).Not()

    sr = (img.select(["SR_B2", "SR_B3", "SR_B4"])
            .multiply(2.75e-05)
            .add(-0.2))

    return img.addBands(sr, overwrite=True).updateMask(mask)


def landsat2015_rgb(region, start="2015-01-01", end="2015-12-31", max_cloud=80):
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
             .filterDate(start, end)
             .filterBounds(region)
             .filter(ee.Filter.lte("CLOUD_COVER", max_cloud))
             .map(mask_l8_sr))

    # Median RGB composite (R=SR_B4, G=SR_B3, B=SR_B2)
    comp = col.select(["SR_B4", "SR_B3", "SR_B2"]).median()


    return comp.visualize(min=0.0, max=0.3)

In [ ]:
CHIP_SIZE_PX = 256
BUFFER_M = 1280
PER_CLUSTER = 10
MAX_POINTS = 20000
BASE_SLEEP_S = 0.4
SAVE_EVERY = 200

SAMPLED_CLUSTERS_CSV = os.path.join(OUT_DIR, "sampled_clusters.csv")

def point_region(lon, lat, buffer_m=BUFFER_M):
    p = ee.Geometry.Point([float(lon), float(lat)])
    return p.buffer(buffer_m).bounds()

def thumb_png_url(img, region, size_px=CHIP_SIZE_PX):
    return img.getThumbURL({
        "region": region,
        "dimensions": f"{size_px}x{size_px}",
        "format": "png"
    })

def safe_get(url, timeout=60, max_retries=6):
    backoff = 1.0
    resp = None

    for _ in range(max_retries):
        try:
            resp = requests.get(url, timeout=timeout)

            if resp.status_code in (429, 500, 503):
                time.sleep(backoff)
                backoff = min(backoff * 2, 30)
                continue

            return resp

        except Exception:
            time.sleep(backoff)
            backoff = min(backoff * 2, 30)

    return resp

# Fixed random cluster sample
all_clusters = locs["cluster_id"].drop_duplicates()
n_clusters = min(MAX_POINTS // PER_CLUSTER, len(all_clusters))

if os.path.exists(SAMPLED_CLUSTERS_CSV):
    sampled_clusters = pd.read_csv(SAMPLED_CLUSTERS_CSV)["cluster_id"]
else:
    sampled_clusters = all_clusters.sample(n=n_clusters, random_state=42)
    pd.DataFrame({"cluster_id": sampled_clusters}).to_csv(SAMPLED_CLUSTERS_CSV, index=False)

# Build subset points
subset = locs[locs["cluster_id"].isin(sampled_clusters)].copy()

subset = (subset
          .sort_values(["cluster_id", "sample_id"])
          .groupby("cluster_id", as_index=False)
          .head(PER_CLUSTER)
          .reset_index(drop=True))

subset = subset.head(MAX_POINTS)

print("Will download:", len(subset),
      "clusters:", subset["cluster_id"].nunique())

# Resume status table
rows = []
if os.path.exists(META_OUT):
    prev = pd.read_csv(META_OUT)
    rows = list(
        prev[["cluster_id", "sample_id", "lat", "lon", "file", "status", "detail"]]
        .itertuples(index=False, name=None)
    )

# Download loop
for r in tqdm(subset.itertuples(index=False), total=len(subset)):
    cid = int(r.cluster_id)
    sid = int(r.sample_id)
    lat = float(r.lat)
    lon = float(r.lon)

    out_name = f"c{cid}_s{sid}_lat{lat:.5f}_lon{lon:.5f}.png"
    out_path = os.path.join(OUT_DIR, out_name)

    # Skip if already downloaded
    if os.path.exists(out_path):
       continue

    try:
        region = point_region(lon, lat)
        img = landsat2015_rgb(region)

        url = thumb_png_url(img, region)
        resp = safe_get(url)

        if resp is not None and resp.status_code == 200 and resp.content:
            with open(out_path, "wb") as f:
                f.write(resp.content)
            rows.append((cid, sid, lat, lon, out_name, "ok", None))
        else:
            status_code = None if resp is None else resp.status_code
            rows.append((cid, sid, lat, lon, out_name, "http_error", status_code))

    except Exception as e:
        rows.append((cid, sid, lat, lon, out_name, "exception", str(e)[:200]))

    # Save progress
    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows, columns=[
            "cluster_id", "sample_id", "lat", "lon", "file", "status", "detail"
        ]).to_csv(META_OUT, index=False)

    time.sleep(BASE_SLEEP_S)

pd.DataFrame(rows, columns=[
    "cluster_id", "sample_id", "lat", "lon", "file", "status", "detail"
]).to_csv(META_OUT, index=False)

In [ ]:
status = pd.read_csv(META_OUT)
print(status["status"].value_counts(dropna=False).head(20))
print("Total rows:", len(status))